In [ ]:
! pip install langchain langchain-community pydantic langchain-core duckduckgo-search ddgs langchain-experimental

# Built in Tools

In [39]:
from langchain_community.tools import DuckDuckGoSearchRun

websearchtool = DuckDuckGoSearchRun()

result = websearchtool.invoke("Top 3 news on geopolitics today")

print(result)

print(websearchtool.name)
print(websearchtool.description)
print(websearchtool.args)

16 hours ago · Independent geopolitics analysis and geopolitical risk coverage for 20 years: armed conflict, supply chains, and the economics driving global risk. 16 hours ago · Get the latest geopolitical news, international affairs insights, and international relations updates from GeoPulse - your trusted source for expert analysis. Today's geo-political landscape is defined by great power competition, a shift towards a multipolar world and a contest over the rules of the international order. Sanctions, tariffs and export controls have become everyday tools of economic statecraft, linking trade directly to security. Top stories Briefing What will a lame-duck Donald Trump get up to overseas? Embattled at home, the president will look abroad. America and China see profit in keeping the whole world on edge. Fear united Europe. Nationalism may tear it apart. Donald Trump and Vladimir Putin made Europe serious about its own defences. That...
duckduckgo_search
A wrapper around DuckDuckGo S

In [14]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

result = shell_tool.invoke("ls -ltr")

print( result )

Executing command:
 ls -ltr
total 4
drwxr-xr-x 1 root root 4096 Nov 18 15:58 sample_data



# Custom tools

## 1) Create Custom tool with Tool Decorator




In [23]:
from langchain_core.tools import tool

In [24]:
#step 1: Create the function that is being used as tool

def division(numerator, denominator):
  """ To divide the numerator by denominator"""
  return numerator/denominator

In [46]:
#Step 2: define the type hints for inputs [Optional but recommended]

def division(numerator:int, denominator:int) -> int:
  """ To divide the numerator by denominator"""
  return numerator/denominator

In [47]:
#Step 3 add the tool decorator on top of function

@tool
def division(numerator:int, denominator:int) -> int:
  """ To divide the numerator by denominator"""
  return numerator/denominator

In [30]:
result = division.invoke({"numerator":25,"denominator":5})
print(result)

5.0


In [42]:
print(division.name)
print(division.description)
print(division.args)

# json schema that will be sent to the LLM model
print(division.args_schema.model_json_schema())

division
To divide the numerator by denominator
{'numerator': {'title': 'Numerator', 'type': 'integer'}, 'denominator': {'title': 'Denominator', 'type': 'integer'}}
{'description': 'To divide the numerator by denominator', 'properties': {'numerator': {'title': 'Numerator', 'type': 'integer'}, 'denominator': {'title': 'Denominator', 'type': 'integer'}}, 'required': ['numerator', 'denominator'], 'title': 'division', 'type': 'object'}


## 2) Create Custom tool with StructuredTool




In [51]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [52]:
class DivisionInputs(BaseModel):
  numerator: int = Field(description="The numerator for division", required=True)
  denominator: int = Field(description="The denominator for division", required=True)

/tmp/ipykernel_516/403992742.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  numerator: int = Field(description="The numerator for division", required=True)
/tmp/ipykernel_516/403992742.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  denominator: int = Field(description="The denominator for division", required=True)


In [53]:
def division_fun(numerator:int, denominator:int) -> int:
  return numerator/denominator

In [54]:
structured_division_tool = StructuredTool.from_function(
    func=division_fun,
    description="To divide the numerator by denominator",
    name="division",
    args_schema=DivisionInputs
)

In [55]:
result = structured_division_tool.invoke({"numerator":25,"denominator":5})
print(result)

5.0


In [56]:
print(structured_division_tool.name)
print(structured_division_tool.description)
print(structured_division_tool.args)

division
To divide the numerator by denominator
{'numerator': {'description': 'The numerator for division', 'title': 'Numerator', 'type': 'integer'}, 'denominator': {'description': 'The denominator for division', 'title': 'Denominator', 'type': 'integer'}}


## 3) Create Custom tool with BaseTool


In [62]:
from langchain_core.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field

In [63]:
class DivisionInputs(BaseModel):
  numerator: int = Field(description="The numerator for division", required=True)
  denominator: int = Field(description="The denominator for division", required=True)

/tmp/ipykernel_516/403992742.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  numerator: int = Field(description="The numerator for division", required=True)
/tmp/ipykernel_516/403992742.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  denominator: int = Field(description="The denominator for division", required=True)


In [64]:
class DivisonTool(BaseTool):
  name : str = "division"
  description : str = "To divide the numerator by denominator"

  args_schema: Type[BaseModel] = DivisionInputs

  def _run(self, numerator:int, denominator:int) -> int:
    return numerator/denominator

In [65]:
division_tool = DivisonTool()

result = division_tool.invoke({"numerator":25,"denominator":5})
print(result)

5.0
